# Dyslexia Handwriting Letter Classifier
This notebook trains a lightweight letter classifier on the Kaggle Synthetic Dyslexia Handwriting Dataset using TensorFlow transfer learning. It is designed to be runnable in Colab with minimal setup.

In [1]:
# Import Required Libraries
import os
import pathlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing import image_dataset_from_directory

print('TensorFlow version:', tf.__version__)


: 

In [2]:
!nvidia-smi

In [3]:
import os

os.environ['KAGGLE_API_TOKEN'] = 'KAGGLE_API_TOKEN'

## Load and Inspect Handwriting Dataset

This notebook expects the Kaggle dataset to be available in the Colab backend runtime. Set `KAGGLE_API_TOKEN` in the runtime environment and use the Kaggle command-line tool to download the dataset. If you are running locally instead, the notebook will fall back to the local workspace path.

In [4]:
!pip install -q --upgrade kaggle

BASE_DIR = pathlib.Path('/content')
DATASET_DIR = BASE_DIR / 'dataset'

if not DATASET_DIR.exists():
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    !kaggle datasets download -d michaelfink0923/synthetic-dyslexia-handwriting-dataset -p "$DATASET_DIR" --unzip

print('Dataset folder:', DATASET_DIR)
!find "$DATASET_DIR" -maxdepth 2 -type f | sed -n '1,20p'

In [5]:
!pip install -q scikit-learn pillow

## Preprocess Images and Labels

This section converts the object-detection dataset into letter-level classification examples. It:
- finds image files and corresponding YOLO-format annotation files
- crops each letter bounding box
- maps the annotation class to a label
- splits data into train, validation, and test sets

In [ ]:
from PIL import Image

LABEL_MAP = {
    0: 'normal',
    1: 'reversal',
    2: 'corrected'
}

RAW_BASE_DIR = DATASET_DIR / 'kaggle' / 'working' / 'synthdata'
RAW_IMAGES_DIR = RAW_BASE_DIR / 'images'
RAW_LABELS_DIR = RAW_BASE_DIR / 'labels'


def load_examples(images_dir, labels_dir):
    examples = []
    for img_path in sorted(images_dir.rglob('*.jpg')) + sorted(images_dir.rglob('*.png')):
        ann_path = labels_dir / f'{img_path.stem}.txt'
        if not ann_path.exists():
            continue
        img = Image.open(img_path).convert('RGB')
        width, height = img.size
        with ann_path.open('r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                class_id, x_center, y_center, w, h = map(float, parts)
                x_center *= width
                y_center *= height
                w *= width
                h *= height
                x1 = max(0, int(x_center - w / 2))
                y1 = max(0, int(y_center - h / 2))
                x2 = min(width, int(x_center + w / 2))
                y2 = min(height, int(y_center + h / 2))
                crop = img.crop((x1, y1, x2, y2))
                examples.append((crop, int(class_id)))
    return examples

train_examples = load_examples(RAW_IMAGES_DIR / 'train', RAW_LABELS_DIR / 'train')
val_examples = load_examples(RAW_IMAGES_DIR / 'val', RAW_LABELS_DIR / 'val')

print('Original train letter examples:', len(train_examples))
print('Original val letter examples:', len(val_examples))

# Inspect sample train crops
sample_count = min(6, len(train_examples))
fig, axes = plt.subplots(1, sample_count, figsize=(14, 4))
for i in range(sample_count):
    crop, cls = train_examples[i]
    axes[i].imshow(crop)
    axes[i].axis('off')
    axes[i].set_title(LABEL_MAP.get(cls, str(cls)))
plt.show()


In [13]:
!ls "$DATASET_DIR/kaggle/working/synthdata"

In [ ]:
IMG_SIZE = 224

# Resize and save cropped letters into a local dataset directory for training.
OUTPUT_DIR = BASE_DIR / 'letter_dataset'
train_dir = OUTPUT_DIR / 'train'
val_dir = OUTPUT_DIR / 'val'
test_dir = OUTPUT_DIR / 'test'

for split_dir in [train_dir, val_dir, test_dir]:
    for label in LABEL_MAP.values():
        (split_dir / label).mkdir(parents=True, exist_ok=True)

np.random.seed(42)
idxs = np.random.permutation(len(val_examples))
val_end = int(0.85 * len(val_examples))

for i, example_idx in enumerate(idxs):
    crop, cls = val_examples[example_idx]
    label = LABEL_MAP.get(cls, 'unknown')
    if i < val_end:
        split = val_dir
        filename = split / label / f'{label}_{i:05d}.png'
    else:
        split = test_dir
        filename = split / label / f'{label}_{i - val_end:05d}.png'
    crop.resize((IMG_SIZE, IMG_SIZE)).save(filename)

for i, (crop, cls) in enumerate(train_examples):
    label = LABEL_MAP.get(cls, 'unknown')
    filename = train_dir / label / f'{label}_{i:05d}.png'
    crop.resize((IMG_SIZE, IMG_SIZE)).save(filename)

print('Saved crops to:', OUTPUT_DIR)

# Class distribution
for split_name, split_path in [('train', train_dir), ('val', val_dir), ('test', test_dir)]:
    counts = {label: len(list((split_path / label).glob('*.png'))) for label in LABEL_MAP.values()}
    print(split_name, counts)

# Visualize distribution
train_counts = [len(list((train_dir / label).glob('*.png'))) for label in LABEL_MAP.values()]
plt.bar(LABEL_MAP.values(), train_counts)
plt.title('Train split class distribution')
plt.ylabel('Count')
plt.show()


## Build a Convolutional Neural Network

We use a lightweight transfer learning backbone with a small input size. MobileNetV3Small is a good fit for low-resource inference while still providing useful feature extraction.

In [8]:
BATCH_SIZE = 32

train_ds = image_dataset_from_directory(
    train_dir,
    label_mode='int',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_ds = image_dataset_from_directory(
    val_dir,
    label_mode='int',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False
)

auto = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=auto)
val_ds = val_ds.prefetch(buffer_size=auto)

preprocess_input = keras.applications.mobilenet_v3.preprocess_input

base_model = keras.applications.MobileNetV3Small(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(LABEL_MAP), activation='softmax')(x)
model = keras.Model(inputs, outputs)

model.summary()


## Compile and Train the Model

Compile with an optimizer and cross-entropy loss. Monitor validation accuracy while training on the letter dataset.

In [9]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint('best_letter_classifier.keras', save_best_only=True)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)

plt.plot(history.history['accuracy'], label='train_accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()


## Evaluate Model Performance

Evaluate on the held-out test set and plot a confusion matrix to inspect per-class behavior.

In [ ]:
test_ds = image_dataset_from_directory(
    test_dir,
    label_mode='int',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = test_ds.prefetch(buffer_size=auto)

loss, accuracy = model.evaluate(test_ds)
print(f'Test loss: {loss:.4f}, test accuracy: {accuracy:.4f}')

# Confusion matrix
from sklearn.metrics import confusion_matrix

y_true = np.concatenate([y.numpy() for x, y in test_ds], axis=0)
y_pred = np.concatenate([np.argmax(model.predict(x), axis=-1) for x, y in test_ds], axis=0)
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(6, 4))
ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.set_title('Confusion Matrix')
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_xticks(range(len(LABEL_MAP)))
ax.set_yticks(range(len(LABEL_MAP)))
ax.set_xticklabels(list(LABEL_MAP.values()), rotation=45)
ax.set_yticklabels(list(LABEL_MAP.values()))
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha='center', va='center', color='black')
plt.tight_layout()
plt.show()


## Make Predictions on New Handwriting Samples

Use the trained model to predict the class of new letter crops and inspect output probabilities.

In [ ]:
def predict_image(img_path):
    img = Image.open(img_path).convert('RGB')
    img_resized = img.resize((IMG_SIZE, IMG_SIZE))
    img_array = np.array(img_resized)[None, ...]
    predictions = model.predict(img_array)
    pred_cls = np.argmax(predictions, axis=-1)[0]
    confidence = float(np.max(predictions))
    return LABEL_MAP[pred_cls], confidence, img_resized

sample_paths = list((test_dir / LABEL_MAP[0]).glob('*.png'))[:3]
fig, axes = plt.subplots(1, len(sample_paths), figsize=(12, 4))
for ax, sample_path in zip(axes, sample_paths):
    label, confidence, img_resized = predict_image(sample_path)
    ax.imshow(img_resized)
    ax.axis('off')
    ax.set_title(f'{label} ({confidence:.2f})')
plt.show()


## Save and Load the Trained Model

Save the best weights and export a lightweight model format for later deployment. TensorFlow Lite is a good option for low-spec devices.

In [ ]:
model.save('letter_classifier_model.keras')
print('Saved letter_classifier_model.keras')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
with open('letter_classifier_model.tflite', 'wb') as f:
    f.write(tflite_model)
print('Saved letter_classifier_model.tflite')

# Verify loading the TFLite model
interpreter = tf.lite.Interpreter(model_path='letter_classifier_model.tflite')
interpreter.allocate_tensors()
print('TFLite model loaded successfully')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
from pathlib import Path

drive_root = Path('/content/drive/MyDrive')
if drive_root.exists():
    drive_root.mkdir(parents=True, exist_ok=True)
    shutil.copy('letter_classifier_model.keras', drive_root / 'letter_classifier_model.keras')
    shutil.copy('letter_classifier_model.tflite', drive_root / 'letter_classifier_model.tflite')
    print('Saved model files to', drive_root)
else:
    print('Google Drive is not mounted at /content/drive. Mount it first to save to Google Drive.')
